# Program - cn_pt

**Purpose**

Read TaiESM hindcast simulation data. Given a variable and lat/lon, plot time-pressure cross-section.

**Content**
- read data
- plot cn_pt

**Author:** Yi-Hsuan Chen (yihsuan@umich.edu)

**Date:** 

**Reference program:**



In [20]:
import matplotlib.pyplot as plt
from matplotlib.ticker import (MultipleLocator, AutoMinorLocator)
import numpy as np
import xarray as xr
import io, os, sys, types

import yhc_module as yhc
#import read_data as read_data   ## on GFDL PP/AN
import read_data_big as read_data    ## on my Mac

#--- silence dask large chunk and silence the warning.
import dask
dask.config.set(**{'array.slicing.split_large_chunks': False})

xr.set_options(keep_attrs=True)  # keep attributes after xarray operation

# Functions

## select_region_ds

In [21]:
##################
##################
##################
def select_region_ds(ds,
                  region="lat_lon",
                  lat=None, lon=None, 
                 ):

    func_name = "select_ds_region"
    
    #--- given the lat/lon range of a region
    if (region == "NE_CA"):
        #lowerlon=235; upperlon=245; lowerlat=28; upperlat=35
        lowerlon=225; upperlon=250; lowerlat=24; upperlat=40

    elif (region == "DYCOMS"): 
        #--- reference: Eyeballing in Fig. 1 in Stevens et al. (2007, MWR)
        region_name = "DYCOMS (29.5-33N, 120-123.5W)"
        lowerlat =  29.5   # 29.5N
        upperlat =  33     # 33N
        lowerlon =  236.5  # 123.5W
        upperlon =  240    # 120W

    elif (region == "lat_lon"): 
        if lat is None or lon is None:
            error_msg = f"ERROR: function [{func_name}] needs lat [{lat}] and lon [{lon}] when region='lat_lon'"
            raise KeyError(error_msg)
    
    else:
        lowerlon=-1000; upperlon=1000; lowerlat=-1000; upperlat=1000

    #--- select for in the given region
    if (region == "lat_lon"):
        lat_index, lon_index = get_lat_lon_indexes(ds, lat, lon)
        ds_region = ds.isel(lat=lat_index, lon=lon_index)

    else:
        lon_slice = slice(lowerlon, upperlon)
        lat_slice = slice(lowerlat, upperlat)
        ds_region = ds.sel(lat=lat_slice, lon=lon_slice)
    
    return ds_region

##################
##################
##################
def get_lat_lon_indexes(ds, lat, lon):
    lat_index = abs(ds['lat'] - lat).argmin().item()
    lon_index = abs(ds['lon'] - lon).argmin().item()

    return lat_index, lon_index

#-----------
# do_test
#-----------

do_test="111"
#do_test=False

if (do_test == "111"):
    choice = "custom_Ttend_3hr"
    icdate = "20010710"
    ds1 = read_data.read_TaiESM1_hindcast_icdate_files(choice, icdate, file_dates=5)
    
    lat=31.5 ; lon=236.5
    
    ds1_region = select_region_ds(ds1, region="lat_lon", lat=lat, lon=lon)

    #ds1_region

## read_hindcast_icdate_lat_lon

In [22]:
def read_hindcast_icdate_lat_lon(icdate, 
    lat=31.5, lon=236.5,
    ):

    file_dates = 5

    #--- read datasets
    choice = "custom_Ttend_3hr"
    ds_Ttend = read_data.read_TaiESM1_hindcast_icdate_files(choice, icdate, file_dates=file_dates)

    choice = "custom_state_3hr"
    ds_state = read_data.read_TaiESM1_hindcast_icdate_files(choice, icdate, file_dates=file_dates)

    choice = "custom_Qtend_3hr"
    ds_Qtend = read_data.read_TaiESM1_hindcast_icdate_files(choice, icdate, file_dates=file_dates)

    #--- get data at the given lat/lon
    ds_Ttend_latlon = select_region_ds(ds_Ttend, region="lat_lon", lat=lat, lon=lon)
    ds_Qtend_latlon = select_region_ds(ds_Qtend, region="lat_lon", lat=lat, lon=lon)
    ds_state_latlon = select_region_ds(ds_state, region="lat_lon", lat=lat, lon=lon)

    return ds_state_latlon, ds_Ttend_latlon, ds_Qtend_latlon

#-----------
# do_test
#-----------

do_test="111"
#do_test=False

if (do_test == "111"):
    icdate = "20010710"
    ds_state_latlon, ds_Ttend_latlon, ds_Qtend_latlon = read_hindcast_icdate_lat_lon(icdate)

#ds_state_latlon
#ds_Ttend_latlon
#ds_Qtend_latlon

# Read data

## Read TaiESM hindcast simulation data

In [23]:
icdate = "20010710"
lat=31.5 ;  lon=236.5
ds_state_latlon, ds_Ttend_latlon, ds_Qtend_latlon = read_hindcast_icdate_lat_lon(icdate, lat=lat, lon=lon)


## Plot - 

In [ ]:
#yhc.lib("fdef")   # check out yhc library